# 08 · Phase B (vLLM): query-side LLM work, LLM methods and agentic runs

Plan sections 4.4, 4.5, 4.7 and 5 (phase B). vLLM holds the GPU; the encoder runs on the **CPU** and only embeds short texts (HyDE passages, agent search queries).

1. **Query-side LLM work** for all 100 questions: HyDE passages (Q1), keyword expansions (Q2) and the engine-type filter (F1). Qwen3.8-27B-INT4, thinking off, temperature 0, JSON-schema output where JSON is needed. Every call is cached in `data/cache/llm.jsonl` under the hash of the full request. The outputs don't depend on the index.
2. **Runs that need the LLM**: Q1, Q2, F1 (gate off and on) and T4, the agent (gate on, 4k / 8k / 16k), on both indexes.
3. **Verification** of all runs (plan 8): reproducibility, no label leakage, one hop.

> **Before running: swap from phase A to phase B.**
> ```bash
> docker compose stop encoder-gpu                  # frees the GPU (and port 8081)
> docker compose up -d qdrant vllm && docker compose up -d --force-recreate encoder         # CPU encoder + vLLM (VLLM_CONFIG=serve.retrieval.yaml)
> docker compose logs -f vllm                      # ~2.5 min to healthy
> ```
> No reranker runs here: X1 and X2 ran in notebook 07 on the GPU encoder.

Times: the query-side calls take ~5–15 s per question; the agent takes ~1–2 h for the six T4 runs.

In [ ]:
from pathlib import Path
import os, sys
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)  # all paths below are relative to the repo root
from dotenv import load_dotenv
load_dotenv(ROOT / ".env")
DATA = ROOT / "data"
RESULTS = ROOT / "results"

In [ ]:
import json, re, subprocess, time
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from csrag import runs
from csrag.corpus import COLLECTIONS, Corpus
from csrag.retrievers import Pipeline, Resources, grid, method_config, prompt_hashes
from csrag.retrievers.query_llm import vllm_healthy

PARAMS_VERSION = "v1"

t0 = time.time()
while not vllm_healthy():
    if time.time() - t0 > 900:
        raise TimeoutError("vLLM not healthy: `docker compose up -d vllm`, then `docker compose logs vllm`")
    time.sleep(10)
print("vLLM healthy after", round(time.time() - t0), "s")
corpus = Corpus(DATA)
res = Resources.connect(corpus, DATA)
health = res.encoder.wait_ready()
print("encoder on", health["device"], "| LLM", res.llm.model)
assert health["device"] == "cpu", "phase B uses the CPU encoder (`docker compose up -d encoder`); the GPU belongs to vLLM"

### KV-cache check (plan 3.1)

The startup log must show a KV cache that holds at least one full 32,768-token request. If it doesn't, set `max-model-len: 24576` in `vllm/serve.retrieval.yaml` and drop the 16k expansion budget.

In [ ]:
try:
    log = subprocess.run(["docker", "logs", "ikincitik-vllm"], capture_output=True, text=True, timeout=60)
    text = log.stdout + log.stderr
    conc = [float(x) for x in re.findall(r"Maximum concurrency for 32,768 tokens per request: ([\d.]+)x", text)]
    kv = re.findall(r"GPU KV cache size: ([\d,]+) tokens", text)
    print("KV cache tokens:", kv[-1] if kv else "?", "| max concurrency at 32768:", conc[-1] if conc else "?")
    assert conc and conc[-1] >= 1.0, "KV cache does not hold one 32768-token request"
except FileNotFoundError:
    print("docker CLI not available here: check `docker compose logs vllm | grep concurrency` by hand")

In [ ]:
questions = [json.loads(l) for l in open(DATA / "eval" / "questions.jsonl", encoding="utf-8")]
texts = [q["question"] for q in questions]
prompts = prompt_hashes()
configs = grid(PARAMS_VERSION, prompts=prompts)
print(res.llm.model, "| prompts:", prompts)

## 1. Query-side LLM work

In [ ]:
rows = []
for q in tqdm(questions):
    passage, r1 = res.llm.hyde(q["question"])
    terms, r2 = res.llm.keywords(q["question"])
    engine, r3 = res.llm.extract_filter(q["question"])
    rows.append({"question_id": q["question_id"], "question": q["question"], "hyde": passage, "terms": terms,
                 "f1_engine_type": engine, "label_engine_type": q["engine_type"], "answerable": q["answerable"],
                 "llm_seconds": r1["seconds"] + r2["seconds"] + r3["seconds"],
                 "prompt_tokens": sum(r["usage"]["prompt_tokens"] for r in (r1, r2, r3)),
                 "completion_tokens": sum(r["usage"]["completion_tokens"] for r in (r1, r2, r3)),
                 "finish": [r["finish_reason"] for r in (r1, r2, r3)]})
out = pd.DataFrame(rows)
print(len(res.llm.cache), "cached LLM calls")
out[["llm_seconds", "prompt_tokens", "completion_tokens"]].describe().loc[["mean", "50%", "max"]]

In [ ]:
truncated = out[out["finish"].apply(lambda f: "length" in f)]
print(len(truncated), "questions with an output cut at max_tokens")
pd.set_option("display.max_colwidth", 200)
out[["question_id", "question", "hyde", "terms"]].head(5)

### F1 extraction vs the `engine_type` label (evaluation only)

F1 never sees the label. The label is only compared here, and the oracle F0 is the one method that uses it. `all` questions should map to `null` (no filter); a wrong `piston`/`turbine` filter removes the gold section from every list.

In [ ]:
pd.crosstab(out["label_engine_type"], out["f1_engine_type"].fillna("null (no filter)"), margins=True)

HyDE passages in dense form (CPU encoder; ~100 passages of ~200 tokens):

In [ ]:
t0 = time.time()
h_emb = res.qemb["hyde"].get(list(out["hyde"]), res.encoder)
print(len(h_emb), "HyDE embeddings in", round(time.time() - t0, 1), "s")

## 2. Runs that need the LLM

Q1, Q2 and F1 with the gate off and on (their LLM outputs come from the cache above), then the agent T4 (gate on, three budgets, both indexes). Each agent step is cached; the agent's `search` embeds its new queries with the CPU encoder.

In [ ]:
def execute(cfgs, desc):
    ids = []
    for cfg in tqdm(cfgs, desc=desc):
        ids.append(runs.execute(cfg, Pipeline(cfg, res).run, questions, res.meta(cfg["index"])))
    return ids

t0 = time.time()
llm_ids = execute([c for c in configs if c["method"] in ("Q1", "Q2", "F1")], "Q1 Q2 F1")
print(len(llm_ids), "runs in", round(time.time() - t0), "s")

In [ ]:
agent_cfgs = [c for c in configs if c["method"] == "T4"]
t0 = time.time()
agent_ids = execute(agent_cfgs, "agent")
print(len(agent_ids), "agent runs in", round((time.time() - t0) / 60), "min")
paths = pd.DataFrame([{"run_id": r, "question_id": s["question_id"], "path": " → ".join(s["agent"]["path"])}
                      for r in agent_ids for s in runs.load_stats(r)])
paths.groupby("run_id")["path"].value_counts().groupby(level=0).head(5).rename("questions").to_frame()

## 3. Verification (plan 8)

* **Reproducibility**: re-running a config reproduces the same `hits.parquet` (temperature 0, cached LLM outputs). H1 needs no LLM; T4 replays its cached steps.
* **No label leakage**: no retriever except F0 reads a label; the interface only receives `question_id` and `text`.
* **One hop**: no context in T3, T2+T3 or T4 holds hop-2 content, and every hop-1 item's source is hop-0 content of the same question.
* **Completeness**: every configuration of the grid has a run (110 for a params version).

In [ ]:
import tempfile
for cfg in [method_config("H1", "win500", True, PARAMS_VERSION, prompts=prompts),
            method_config("T4", "units", True, PARAMS_VERSION, budget=8192, prompts=prompts)]:
    rid = runs.run_id_of(cfg)
    with tempfile.TemporaryDirectory() as tmp:
        runs.execute(cfg, Pipeline(cfg, res).run, questions, res.meta(cfg["index"]), root=tmp)
        same = runs.load_hits(rid, tmp).equals(runs.load_hits(rid))
    print(rid, "reproduced:", same)

In [ ]:
import inspect
assert list(inspect.signature(Pipeline.run).parameters) == ["self", "question_id", "text"]
labels = {q["question_id"]: q["engine_type"] for q in questions}
for m in ("H1", "F1", "T1", "X1"):
    assert Pipeline(method_config(m, "units", True, PARAMS_VERSION, prompts=prompts), res, labels=labels).labels is None
label_fields = ("question_type", "engine_type", "gold_kind", "difficulty", "mentions_section_id", "gold", "answer")
src = "".join(inspect.getsource(__import__(f"csrag.retrievers.{m}", fromlist=["x"]))
              for m in ("agentic", "structural", "rerank", "fusion", "query_llm", "qdrant_search"))
print("label field names in retriever code:", [f for f in label_fields if f'["{f}"]' in src or f"['{f}']" in src])
print("only F0 reads labels: OK")

In [ ]:
bad = []
for rid in runs.list_runs(params_version=PARAMS_VERSION):
    cfg = runs.load_config(rid)["config"]
    ex = cfg.get("expansion") or {}
    if ex.get("rule") not in ("refs", "parent+refs", "agent"):
        continue
    h = runs.load_hits(rid)
    h = h[h["in_context"]]
    if (~h["hop"].isin([0, 1])).any():
        bad.append((rid, "hop > 1"))
    for qid, g in h.groupby("question_id"):
        hop0 = set(g.loc[g["hop"] == 0, "unit_id"])
        src_ = set(g.loc[g["hop"] == 1, "source"])
        if not src_ <= hop0:
            bad.append((rid, qid, src_ - hop0))
print(len(bad), "one-hop violations", bad[:5])

In [ ]:
have = set(runs.list_runs(params_version=PARAMS_VERSION))
missing = [runs.run_id_of(c) for c in configs if runs.run_id_of(c) not in have]
print(f"{len(configs) - len(missing)} of {len(configs)} {PARAMS_VERSION} runs present; missing: {missing[:10]}")
pd.Series([runs.load_config(r)["retriever"] for r in have]).value_counts().sort_index().rename("runs").to_frame().T

## v2: the one revision (plan 4.8)

After reading **all** v1 metrics in notebook 09, revise the parameters **once**, by hand, in `src/csrag/retrievers/base.py`:

```python
PARAMS["v2"] = revise("v1", gate_a=GateParams(tau=..., delta=..., min_keep=..., max_keep=...), ...)
```

Then set `PARAMS_VERSION = "v2"` and run phase A (notebook 07, GPU encoder, vLLM stopped) and phase B (this notebook) again. Both versions are kept: v1 is the a-priori result, v2 the revised one (chosen after seeing v1 on the same questions).